In [0]:
# Configuração
storage_account = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-account-name')
storage_key     = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-key-bitcoin')

LANDING_PATH = f'abfss://landing@{storage_account}.dfs.core.windows.net/'

In [0]:

# Lê o JSON bruto
import json
from azure.storage.blob import BlobServiceClient
from io import StringIO

blob_service_client = BlobServiceClient(
    account_url=f'https://{storage_account}.blob.core.windows.net',
    credential=storage_key
)

blob_client = blob_service_client.get_blob_client(container='landing', blob='bitcoin_daily.json')
content = blob_client.download_blob().readall()
data = json.loads(content)

print(f'Chaves do JSON: {list(data.keys())}')
print(f'Total de registros prices: {len(data["prices"])}')
print(f'Total de registros market_caps: {len(data["market_caps"])}')
print(f'Total de registros total_volumes: {len(data["total_volumes"])}')
print(f'\nPrimeiro registro prices: {data["prices"][0]}')
print(f'Último registro prices: {data["prices"][-1]}')

In [0]:

from pyspark.sql import functions as F

records = []
for i in range(len(data['prices'])):
    records.append({
        'timestamp_ms': data['prices'][i][0],
        'preco_usd':    data['prices'][i][1],
        'market_cap':   data['market_caps'][i][1],
        'volume':       data['total_volumes'][i][1]
    })

df = spark.createDataFrame(records)
df = df.withColumn('data', F.to_date(F.from_unixtime(F.col('timestamp_ms') / 1000)))

df.printSchema()
df.orderBy('data').display()

In [0]:

print(f'Total de registros: {df.count()}')
print(f'Data mínima: {df.agg(F.min("data")).collect()[0][0]}')
print(f'Data máxima: {df.agg(F.max("data")).collect()[0][0]}')
print(f'Preço mínimo USD: R$ {df.agg(F.min("preco_usd")).collect()[0][0]:,.2f}')
print(f'Preço máximo USD: R$ {df.agg(F.max("preco_usd")).collect()[0][0]:,.2f}')

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15, 5))
sns.lineplot(x='data', y='preco_usd', data=df.orderBy('data').toPandas())
plt.title('Bitcoin Price')
plt.xlabel('Data')
plt.ylabel('Preço USD')
plt.show()

In [0]:
import plotly.express as px

fig = px.line(df.orderBy('data').toPandas(), x='data', y='preco_usd', title='Bitcoin Price')
fig.update_xaxes(rangeslider_visible=True)
fig.show()